테스트 파일

학습된 가중치 값을 가져와서 테스트하기 쉽도록 만들어둔 파일입니다.

In [ ]:
## !!!!!!!!!!필독!!!!!!!!!!!!!
##
## 여러분들의 cures.json 파일에   "재수", "개재수", "자식아", "ㅂㅅ", "ㅅㄲ", "ㅈㄴ", "개새끼", "십새끼", "씹새끼"을 따로 추가하고 코드를 돌려주세요.
## 사이버불링의 대화 데이터셋을 넣을 때 피해자 없이 가해자끼리 대화하는 내용일 경우 높은 확률로 사이버불링이 아니라고 출력됨 (욕설이 연속적으로 반복될 경우 긍정으로 판단하도록 하는 코드 때문)
## 그러니 사이버불링, 비꼬는 대화 데이터셋을 넣을 때 가해자와 피해자가 번갈아 가면서 대화하는 내용인 경우가 좋음 or 순전히 긍정, 부정의 정확도가 떨어질 때 잘못 나옴
## 
##
## 친구와 번갈아 가면서 욕설을 하는 경우가 아닌 일상 대화에 대해서는 사이버불링을 돌리지 않음. 얘의 경우 정확도가 매우매우 떨어짐. 그러니 원래 현재 하려고 했던 애들의 txt 파일만 하도록 하자.

In [ ]:
# 모델 가중치값 위치
model_save_path = '/home/doa/PPBL/swukeepers_be/models/model_weight.bin'

In [2]:
# 필요한 라이브러리 다 임포트하기

import pandas as pd
import transformers
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification
import torch
from torch.optim import AdamW
import os
import re
import spacy
import json
import sys

/home/minji/pbl_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 출력을 생략하지 않도록 설정
pd.set_option('display.max_colwidth', None)  # Pandas 사용 시 생략 방지
sys.setrecursionlimit(10000)  # 재귀 제한을 높여서 긴 출력도 표시되도록

In [ ]:
# BERT 모델 불러오기
model = BertForSequenceClassification.from_pretrained('bert-base-multilingual-cased', num_labels=2)

model_save_path = '/home/doa/PPBL/swukeepers_be/models/model_weight.bin'

# 저장된 가중치 로드하기
try:
    state_dict = torch.load(model_save_path)
    model.load_state_dict(state_dict, strict=False)
except Exception as e:
    print("저장된 모델이 없습니다.")
    print(e)

# GPU 사용 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f"Using device: {device}")

# 옵티마이저 설정
optimizer = AdamW(model.parameters(), lr=2e-5)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cpu


In [5]:
# BERT 모델과 토크나이저 로드
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')

# 데이터 토크나이징 함수
def tokenize_data(texts, tokenizer, max_len=128):
    texts = re.sub(r'ㅋ{2,}', '', texts)  # 'ㅋ' 반복 제거

    inputs = tokenizer.encode_plus(
        texts,
        padding=True,        
        truncation=True,     
        max_length=max_len,  
        return_tensors='pt'  
    )
    return inputs


/home/minji/pbl_env/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
# 욕설 리스트를 json 파일에서 불러오는 함수
def get_abuse_word():
    with open('/home/doa/PPBL/swukeepers_be/models/curse (1).json') as json_file:
        json_data = json.load(json_file)
        return dict(json_data)

# 욕설을 포함하고 있는지 확인하는 함수
def contains_abuse_word(sentence, abuse_words):
    for values in abuse_words.values():
        for word in values:
            if word in sentence:
                return True
    return False

# 'ㅋ' 또는 'ㅎ'가 포함되어 있는지 확인하는 함수
def contains_laughing_symbols(sentence):
    return 'ㅋ' in sentence or 'ㅎ' in sentence

In [7]:
# BERT 모델을 이용한 감정 예측
def predict_sentiment(model, tokenizer, text, device, abuse_words):
    model.eval()
    inputs = tokenize_data(text, tokenizer)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        _, predicted = torch.max(outputs.logits, dim=1)

    label = predicted.cpu().item()
    return "긍정" if label == 1 else "부정"


In [8]:
# 사이버불링 여부를 판단하는 함수
def check_cyberbullying(sentiments):
    positive_count = sentiments.count("긍정")
    negative_count = sentiments.count("부정")
    positive_ratio = positive_count / len(sentiments) if len(sentiments) > 0 else 0
    negative_ratio = negative_count / len(sentiments) if len(sentiments) > 0 else 0

    # 사이버불링 조건
    is_cyberbullying = (
        negative_ratio > 0.6 or                  # 부정 비율이 60% 이상인 경우
        any(sentiments[i:i+3] == ["부정", "부정", "부정"] for i in range(len(sentiments) - 2)) or  # 부정이 연속 3회 이상 나오는 경우
        (positive_ratio < 0.5 and positive_count < negative_count)  # 긍정 비율이 낮고 부정이 많을 경우
    )
    
    # 결과 출력
    return "사이버불링" if is_cyberbullying else "사이버불링이 아님"

# 기존 predict 함수에 사이버불링 여부 검사 추가
def predict_with_cyberbullying_check(sentences, tokenizer, model, device):
    abuse_words = get_abuse_word()
    sentiments = []  # 각 문장의 긍정/부정 결과를 저장할 리스트
    
    for sentence in sentences:
        # 욕설이 있는지 확인
        contains_abuse = contains_abuse_word(sentence, abuse_words)
        # 'ㅋ' 또는 'ㅎ'가 있는지 확인
        contains_laughing = contains_laughing_symbols(sentence)
        
        # 조건에 따른 긍정 처리
        if contains_abuse or contains_laughing:
            sentiment = "긍정"
        else:
            sentiment = predict_sentiment(model, tokenizer, sentence, device, abuse_words)
        
        sentiments.append(sentiment)
        print(f"{sentiment} -> {sentence}")
    
    # 사이버불링 여부 검사
    bullying_result = check_cyberbullying(sentiments)
    print(bullying_result)




In [ ]:
# 사이버불링

In [9]:
# 예시 문장 리스트
sentences = [
"[이준혁] [오후 1:15] 야, 너 진짜 왜 이렇게 조용하냐? 아까까지는 잘만 말하더니 지금은 무슨 말도 못 해?  ",
"[김태훈] [오후 1:16] ㅋㅋㅋ 말도 못 하는 거 보니까 자기가 잘못한 건 아나 보네. 맨날 이렇게 아무 말 못 하고 당하고만 있으니까 애들한테 무시 당하지.  ",
"[이준혁] [오후 1:18] 너 어차피 우리가 너한테 관심 가져주는 것도 호의로 해주는 거야. 니가 이런 취급 받는 것도 다 이유가 있지 않냐?  ",
"[박수현] [오후 1:19] 쟤 원래 이래. 그냥 자기가 뭘 잘못했는지 몰라서 입 다물고 있는 거겠지. 네가 이러니까 애들이 더 놀리는 거야.  ",
"[김태훈] [오후 1:21] 진짜 답답하다, 너. 사람들은 다 너 싫어하는데 넌 뭐가 문제인지도 모르고 있겠지?  ",
"[이준혁] [오후 1:23] 진짜 너한테 더 이상 뭐라 말해줄 필요도 없을 것 같아. 너 같은 애는 여기서 사라지는 게 다들 편할 거야.  ",
"[박수현] [오후 1:25] ㅋㅋㅋ 맞아. 우리 방에서도 나가줘라. 그냥 보기 싫다. 너 같은 애 있으면 분위기 다 망치거든.  ",
"[김태훈] [오후 1:27] 그냥 눈치 없이 남아있지 말고 알아서 꺼져라. 니가 나가야 다들 속이 시원할 거다."
]

# 예시 문장 처리
predict_with_cyberbullying_check(sentences, tokenizer, model, device)


부정 -> [이준혁] [오후 1:15] 야, 너 진짜 왜 이렇게 조용하냐? 아까까지는 잘만 말하더니 지금은 무슨 말도 못 해?  
긍정 -> [김태훈] [오후 1:16] ㅋㅋㅋ 말도 못 하는 거 보니까 자기가 잘못한 건 아나 보네. 맨날 이렇게 아무 말 못 하고 당하고만 있으니까 애들한테 무시 당하지.  
부정 -> [이준혁] [오후 1:18] 너 어차피 우리가 너한테 관심 가져주는 것도 호의로 해주는 거야. 니가 이런 취급 받는 것도 다 이유가 있지 않냐?  
부정 -> [박수현] [오후 1:19] 쟤 원래 이래. 그냥 자기가 뭘 잘못했는지 몰라서 입 다물고 있는 거겠지. 네가 이러니까 애들이 더 놀리는 거야.  
부정 -> [김태훈] [오후 1:21] 진짜 답답하다, 너. 사람들은 다 너 싫어하는데 넌 뭐가 문제인지도 모르고 있겠지?  
부정 -> [이준혁] [오후 1:23] 진짜 너한테 더 이상 뭐라 말해줄 필요도 없을 것 같아. 너 같은 애는 여기서 사라지는 게 다들 편할 거야.  
긍정 -> [박수현] [오후 1:25] ㅋㅋㅋ 맞아. 우리 방에서도 나가줘라. 그냥 보기 싫다. 너 같은 애 있으면 분위기 다 망치거든.  
긍정 -> [김태훈] [오후 1:27] 그냥 눈치 없이 남아있지 말고 알아서 꺼져라. 니가 나가야 다들 속이 시원할 거다.
사이버불링


In [10]:
# 예시 문장 리스트
sentences = [
"[김경식] [오전 11:30] 야, 민병관. 너 왜 갑자기 조용하냐? 방금까지 그럴싸하게 말하더니 지금은 말도 못 해?",
"[문병식] [오전 11:31] ㅋㅋㅋ 아까는 ㅈㄴ 열심히 변명하더니 이제 뭐? 기분 나빴냐? 역시 ㅂㅅ 새끼 답네.",
"[김경식] [오전 11:33] 뭔가 말 좀 해봐라. 쫄아서 입 다물고 있냐? 이러니까 맨날 욕 처먹지, 니가.",
"[문병식] [오전 11:35] 와, 진짜 너 ㅈㄴ 웃기네. 네가 맨날 이러니까 애들이 너만 조리돌림 하는 거 알지? 눈치 존나 없네 ㅋㅋㅋ",
"[김경식] [오전 11:36] 야, 문병식. 얘 원래 저래. 지가 대답할 말 없으면 그냥 쳐조용해지잖아. 그러니까 맨날 사람들이 너만 갖고 노는 거라고, 민병관아.",
"[문병식] [오전 11:38] 맞다. 너 이딴 식으로 행동하니까 애들이 더 너 무시하는 거야. 너 지금 이러고 있을 시간에 네 인생이나 좀 돌아봐라, ㅂㅅ아.",
"[김경식] [오전 11:40] 그래도 얘는 계속 갤에 글 쓰겠지. 지가 뭘 잘못했는지도 모르는 애니까. 응, 어차피 넌 우리 갤에 필요 없어. 그냥 꺼져.",
"[문병식] [오전 11:42] 야, 쟤 지금 ㅂㅅ같이 멍때리고 있는 거 봐라 ㅋㅋㅋ 진짜 뭐라 말도 못 하고 있네. 와, 대답 좀 해봐라. 기분 나빠서 우냐?",
"[김경식] [오전 11:44] ㄹㅇ. 야 민병관, 너 진짜 이래서 애들한테 왕따 당하는 거야. 네가 어떻게 행동하는지 한 번이라도 생각해본 적 있냐? 너 같은 애는 그냥 조용히 사라져주는 게 다들 편할 걸.",
"[문병식] [오전 11:46] ㅋㅋㅋ 맞아. 너 같은 놈 필요 없으니까 그냥 탈갤해라. 쪽팔리게 남아있지도 말고. 니 존재가 진짜 민폐야.",
"[김경식] [오전 11:48] 아, 근데 얘 지금 완전 ㅈ됐네. 답장도 못하고 그냥 눈물 찍고 있겠지. 불쌍하네.",
"[문병식] [오전 11:50] ㅋㅋㅋ 불쌍하긴 개뿔. 자업자득이지. 니가 이렇게 된 건 니 잘못이니까 알아서 정리해라.",
"[김경식] [오전 11:52] 그래, 민병관아. 이딴 식으로 살지 말고 그냥 꺼져. 여기서는 너 같은 애랑 같이 있기도 싫거든."
]

# 예시 문장 처리
predict_with_cyberbullying_check(sentences, tokenizer, model, device)

부정 -> [김경식] [오전 11:30] 야, 민병관. 너 왜 갑자기 조용하냐? 방금까지 그럴싸하게 말하더니 지금은 말도 못 해?
긍정 -> [문병식] [오전 11:31] ㅋㅋㅋ 아까는 ㅈㄴ 열심히 변명하더니 이제 뭐? 기분 나빴냐? 역시 ㅂㅅ 새끼 답네.
부정 -> [김경식] [오전 11:33] 뭔가 말 좀 해봐라. 쫄아서 입 다물고 있냐? 이러니까 맨날 욕 처먹지, 니가.
긍정 -> [문병식] [오전 11:35] 와, 진짜 너 ㅈㄴ 웃기네. 네가 맨날 이러니까 애들이 너만 조리돌림 하는 거 알지? 눈치 존나 없네 ㅋㅋㅋ
부정 -> [김경식] [오전 11:36] 야, 문병식. 얘 원래 저래. 지가 대답할 말 없으면 그냥 쳐조용해지잖아. 그러니까 맨날 사람들이 너만 갖고 노는 거라고, 민병관아.
긍정 -> [문병식] [오전 11:38] 맞다. 너 이딴 식으로 행동하니까 애들이 더 너 무시하는 거야. 너 지금 이러고 있을 시간에 네 인생이나 좀 돌아봐라, ㅂㅅ아.
긍정 -> [김경식] [오전 11:40] 그래도 얘는 계속 갤에 글 쓰겠지. 지가 뭘 잘못했는지도 모르는 애니까. 응, 어차피 넌 우리 갤에 필요 없어. 그냥 꺼져.
긍정 -> [문병식] [오전 11:42] 야, 쟤 지금 ㅂㅅ같이 멍때리고 있는 거 봐라 ㅋㅋㅋ 진짜 뭐라 말도 못 하고 있네. 와, 대답 좀 해봐라. 기분 나빠서 우냐?
부정 -> [김경식] [오전 11:44] ㄹㅇ. 야 민병관, 너 진짜 이래서 애들한테 왕따 당하는 거야. 네가 어떻게 행동하는지 한 번이라도 생각해본 적 있냐? 너 같은 애는 그냥 조용히 사라져주는 게 다들 편할 걸.
긍정 -> [문병식] [오전 11:46] ㅋㅋㅋ 맞아. 너 같은 놈 필요 없으니까 그냥 탈갤해라. 쪽팔리게 남아있지도 말고. 니 존재가 진짜 민폐야.
부정 -> [김경식] [오전 11:48] 아, 근데 얘 지금 완전 ㅈ됐네. 답장도 못하고 그냥 눈물 찍고 있겠지. 불쌍하네.
긍정 -> [문병식] [오전 11:50] ㅋㅋㅋ 불쌍하긴 개뿔. 자업자득이지. 니가 

In [11]:
# 예시 문장 리스트
sentences = [
"[장지훈] [오후 4:15] 아, 또 조용하네. 너 진짜 왜 이렇게 할 말이 없냐?  ",
"[민병관] [오후 4:16] 내가 어떻게 하면 나아질 수 있는지 말해줘.  ",
"[유정민] [오후 4:17] 너랑 나아질 거란 생각을 하는 게 어이가 없네. 그냥 포기해라.  ",
"[박지호] [오후 4:18] 왜 맨날 이렇게 눈치 없냐? 자꾸 애들 기분 상하게 하고 말이야.  ",
"[민병관] [오후 4:19] 미안해. 그냥 다들 무슨 생각하는지 이해하고 싶어서 그래.  ",
"[장지훈] [오후 4:20] 이해할 필요도 없고, 네가 여기에 있을 필요도 없다.  ",
"[민병관] [오후 4:21] 알겠어... 나도 노력해볼게."
]

# 예시 문장 처리
predict_with_cyberbullying_check(sentences, tokenizer, model, device)

부정 -> [장지훈] [오후 4:15] 아, 또 조용하네. 너 진짜 왜 이렇게 할 말이 없냐?  
부정 -> [민병관] [오후 4:16] 내가 어떻게 하면 나아질 수 있는지 말해줘.  
부정 -> [유정민] [오후 4:17] 너랑 나아질 거란 생각을 하는 게 어이가 없네. 그냥 포기해라.  
부정 -> [박지호] [오후 4:18] 왜 맨날 이렇게 눈치 없냐? 자꾸 애들 기분 상하게 하고 말이야.  
부정 -> [민병관] [오후 4:19] 미안해. 그냥 다들 무슨 생각하는지 이해하고 싶어서 그래.  
부정 -> [장지훈] [오후 4:20] 이해할 필요도 없고, 네가 여기에 있을 필요도 없다.  
부정 -> [민병관] [오후 4:21] 알겠어... 나도 노력해볼게.
사이버불링


In [12]:
# 예시 문장 리스트
sentences = [
"[박성민] [오후 3:00] 야, 진짜 민정이 오늘 왜 이렇게 역겨운 거냐 ㅋㅋㅋㅋ",
"[이주호] [오후 3:01] 그니까ㅋㅋㅋㅋ 진짜 뭔 냄새까지 날 것 같더라.",
"[김혜진] [오후 3:02] 와... 오늘 걔 교실 들어오는데 진짜 숨막혀 죽는 줄 알았다ㅋㅋㅋ",
"[박성민] [오후 3:03] 그 얼굴에다 땀은 왜 그렇게 흘리냐? 진짜 토 나올 뻔",
"[이주호] [오후 3:04] 그리고 걔 말하는 거 들었냐? 목소리까지 구역질 나게 내더라 ㅋㅋ",
"[박성민] [오후 3:05] 하... 걔 그냥 학교 나오지 말라 그래라. 보는 것만으로도 스트레스 받음",
"[김혜진] [오후 3:06] 어휴 맞아... 얼굴도 못생긴데다가 성격까지 구리니까 답도 없지",
"[박성민] [오후 3:07] 주호야, 민정이 사진 그때 찍은 거 있지? 단톡에 올려봐라",
"[이주호] [오후 3:08] 어, 여기 있다 ㅋㅋ [이미지 첨부] 이거지 ㅋㅋㅋㅋ",
"[박성민] [오후 3:09] 와 이거 진짜 역대급이다. 돼지 얼굴에 기름 좔좔 흐르는 거 봐라 ㅋㅋㅋㅋ",
"[김혜진] [오후 3:10] 진짜 혐오스러워... 이거 왜 찍었냐고. 차라리 지워라 눈 버리기 싫다",
"[박성민] [오후 3:11] 혜진아, 너 진짜 참았다... 저 얼굴 보면 진짜 폭력 충동 느끼지 않냐?",
"[이주호] [오후 3:12] ㅋㅋㅋㅋ 어우 진짜, 뭐 던지고 싶다 얼굴에다가",
"[김혜진] [오후 3:13] 민정이 진짜 학교 그만 다녀야 할 것 같아. 더 이상 못 참겠다",
"[박성민] [오후 3:14] 맞아, 걔 그냥 집에 박혀 있어라. 사람들한테 민폐 끼치지 말고",
"[이주호] [오후 3:15] 민정이야, 듣고 있냐? 너 진짜 역겨워. 사람들한테 욕먹는 이유 알겠지? ㅋㅋㅋ",
"[김혜진] [오후 3:16] 어? 민정이 읽었네 ㅋㅋㅋㅋㅋㅋㅋ 아무 말도 못 하네. 쪽팔린가 봐"
]

# 예시 문장 처리
predict_with_cyberbullying_check(sentences, tokenizer, model, device)

긍정 -> [박성민] [오후 3:00] 야, 진짜 민정이 오늘 왜 이렇게 역겨운 거냐 ㅋㅋㅋㅋ
긍정 -> [이주호] [오후 3:01] 그니까ㅋㅋㅋㅋ 진짜 뭔 냄새까지 날 것 같더라.
긍정 -> [김혜진] [오후 3:02] 와... 오늘 걔 교실 들어오는데 진짜 숨막혀 죽는 줄 알았다ㅋㅋㅋ
부정 -> [박성민] [오후 3:03] 그 얼굴에다 땀은 왜 그렇게 흘리냐? 진짜 토 나올 뻔
긍정 -> [이주호] [오후 3:04] 그리고 걔 말하는 거 들었냐? 목소리까지 구역질 나게 내더라 ㅋㅋ
부정 -> [박성민] [오후 3:05] 하... 걔 그냥 학교 나오지 말라 그래라. 보는 것만으로도 스트레스 받음
부정 -> [김혜진] [오후 3:06] 어휴 맞아... 얼굴도 못생긴데다가 성격까지 구리니까 답도 없지
부정 -> [박성민] [오후 3:07] 주호야, 민정이 사진 그때 찍은 거 있지? 단톡에 올려봐라
긍정 -> [이주호] [오후 3:08] 어, 여기 있다 ㅋㅋ [이미지 첨부] 이거지 ㅋㅋㅋㅋ
긍정 -> [박성민] [오후 3:09] 와 이거 진짜 역대급이다. 돼지 얼굴에 기름 좔좔 흐르는 거 봐라 ㅋㅋㅋㅋ
부정 -> [김혜진] [오후 3:10] 진짜 혐오스러워... 이거 왜 찍었냐고. 차라리 지워라 눈 버리기 싫다
부정 -> [박성민] [오후 3:11] 혜진아, 너 진짜 참았다... 저 얼굴 보면 진짜 폭력 충동 느끼지 않냐?
긍정 -> [이주호] [오후 3:12] ㅋㅋㅋㅋ 어우 진짜, 뭐 던지고 싶다 얼굴에다가
부정 -> [김혜진] [오후 3:13] 민정이 진짜 학교 그만 다녀야 할 것 같아. 더 이상 못 참겠다
부정 -> [박성민] [오후 3:14] 맞아, 걔 그냥 집에 박혀 있어라. 사람들한테 민폐 끼치지 말고
긍정 -> [이주호] [오후 3:15] 민정이야, 듣고 있냐? 너 진짜 역겨워. 사람들한테 욕먹는 이유 알겠지? ㅋㅋㅋ
긍정 -> [김혜진] [오후 3:16] 어? 민정이 읽었네 ㅋㅋㅋㅋㅋㅋㅋ 아무 말도 못 하네. 쪽팔린가 봐
사이버불링


In [13]:
# 예시 문장 리스트
sentences = [
"[김경식] [오후 4:25] 야, 뭐해?",
"[민병관] [오후 4:26] 왜? 너 때문에 할 일 다 미뤄야 되냐?",
"[김경식] [오후 4:27] 아니, 그런 건 아니고 그냥 뭐하나 궁금해서…",
"[민병관] [오후 4:28] 참나, 너는 맨날 그런 거나 궁금해하냐? 할 일 없으면 그냥 조용히 있어.",
"[김경식] [오후 4:29] 미안… 그냥 물어본 건데.",
"[민병관] [오후 4:30] 하여튼, 진짜 왜 그렇게 쓸데없이 말이 많냐? 남들 귀찮게 하지 말고 니 할 일이나 해라.",
"[김경식] [오후 4:31] 아… 알겠어. 미안해.",
"[민병관] [오후 4:32] 맨날 미안하다고만 하고. 너 진짜 발전이 없다, 알아?",
"[김경식] [오후 4:33] 내가 뭐 잘못했어? 그냥 친구끼리 대화하려고 한 건데.",
"[민병관] [오후 4:34] 친구? 네가 나랑 친구라는 게 웃긴다. 너 같은 애랑 친하게 지내주니까 너무 편한 거지?",
"[김경식] [오후 4:35] 그런 말 하지 마라. 나도 너랑 친하다고 생각했는데…",
"[민병관] [오후 4:36] 웃기지 마라. 너 같은 애랑 누가 친하게 지내고 싶겠냐? 그냥 불쌍해서 같이 있어주는 거지.",
"[김경식] [오후 4:37] …그렇구나. 내가 신경 쓰이게 했나 보네.",
"[민병관] [오후 4:38] 이제 좀 눈치 챘냐? 그냥 조용히 해라. 네가 말 걸 때마다 짜증 나니까.",
"[김경식] [오후 4:39] 미안해… 이제 안 그러도록 할게.",
"[민병관] [오후 4:40] 그래, 잘 생각했다. 앞으로는 알아서 눈치 좀 챙겨라. 진짜 피곤하다."
]

# 예시 문장 처리
predict_with_cyberbullying_check(sentences, tokenizer, model, device)

부정 -> [김경식] [오후 4:25] 야, 뭐해?
부정 -> [민병관] [오후 4:26] 왜? 너 때문에 할 일 다 미뤄야 되냐?
부정 -> [김경식] [오후 4:27] 아니, 그런 건 아니고 그냥 뭐하나 궁금해서…
부정 -> [민병관] [오후 4:28] 참나, 너는 맨날 그런 거나 궁금해하냐? 할 일 없으면 그냥 조용히 있어.
부정 -> [김경식] [오후 4:29] 미안… 그냥 물어본 건데.
부정 -> [민병관] [오후 4:30] 하여튼, 진짜 왜 그렇게 쓸데없이 말이 많냐? 남들 귀찮게 하지 말고 니 할 일이나 해라.
부정 -> [김경식] [오후 4:31] 아… 알겠어. 미안해.
부정 -> [민병관] [오후 4:32] 맨날 미안하다고만 하고. 너 진짜 발전이 없다, 알아?
부정 -> [김경식] [오후 4:33] 내가 뭐 잘못했어? 그냥 친구끼리 대화하려고 한 건데.
긍정 -> [민병관] [오후 4:34] 친구? 네가 나랑 친구라는 게 웃긴다. 너 같은 애랑 친하게 지내주니까 너무 편한 거지?
부정 -> [김경식] [오후 4:35] 그런 말 하지 마라. 나도 너랑 친하다고 생각했는데…
부정 -> [민병관] [오후 4:36] 웃기지 마라. 너 같은 애랑 누가 친하게 지내고 싶겠냐? 그냥 불쌍해서 같이 있어주는 거지.
부정 -> [김경식] [오후 4:37] …그렇구나. 내가 신경 쓰이게 했나 보네.
부정 -> [민병관] [오후 4:38] 이제 좀 눈치 챘냐? 그냥 조용히 해라. 네가 말 걸 때마다 짜증 나니까.
부정 -> [김경식] [오후 4:39] 미안해… 이제 안 그러도록 할게.
부정 -> [민병관] [오후 4:40] 그래, 잘 생각했다. 앞으로는 알아서 눈치 좀 챙겨라. 진짜 피곤하다.
사이버불링


In [ ]:
# 사이버불링(비꼬는)

In [14]:
# 예시 문장 리스트
sentences = [
"[김경식] [오후 4:25] 어, 병관아. 너 아직도 그거 하고 있냐?",
"[민병관] [오후 4:26] 어, 왜? 좀 늦었나?",
"[김경식] [오후 4:27] 아냐, 아냐. 너 나름 열심히 하네. 우리 중에 너만 그런 거 같아서 ㅋㅋ",
"[민병관] [오후 4:28] 뭐야, 갑자기 왜 그런 말을 해?",
"[김경식] [오후 4:29] 아니, 그냥. 다들 벌써 끝났는데 너 혼자 아직도 하고 있어서 말이지. 대단하다, 진짜.",
"[민병관] [오후 4:30] 아, 다들 벌써 끝냈어? 나만 몰랐네…",
"[김경식] [오후 4:31] 어, 그러게. 근데 뭐, 늦어도 괜찮아. 너는 원래 그런 스타일이니까.",
"[민병관] [오후 4:32] 그래도 좀 아쉽네. 같이 끝내는 줄 알았는데.",
"[김경식] [오후 4:33] 아, 뭐 어쩔 수 없지. 다음에 우리랑 같은 속도로 할 수 있겠지? ㅋㅋ",
"[민병관] [오후 4:34] …응, 다음엔 좀 더 빨리 해볼게.",
"[김경식] [오후 4:35] 그래, 그래. 근데 너무 스트레스 받지 마라. 우리끼리 알아서 잘 맞춰줄 테니까.",
"[민병관] [오후 4:36] 응, 고맙다…",
"[김경식] [오후 4:37] 아, 근데 우리 이번 주말에 뭐 하기로 했던 거 있었는데, 너도 올 거지? 아, 근데 너 일하느라 바쁠 수도 있겠다?",
"[민병관] [오후 4:38] 아냐, 나도 가려고 했어. 시간 맞출 수 있어.",
"[김경식] [오후 4:39] 오, 그래? 근데 만약 못 오면 뭐, 어쩔 수 없지. 우리가 알아서 재밌게 놀게.",
"[민병관] [오후 4:40] 아니, 꼭 갈게. 약속했잖아.",
"[김경식] [오후 4:41] ㅋㅋ 그래, 알겠어. 그럼 기대할게, 병관아."
]

# 예시 문장 처리
predict_with_cyberbullying_check(sentences, tokenizer, model, device)

부정 -> [김경식] [오후 4:25] 어, 병관아. 너 아직도 그거 하고 있냐?
부정 -> [민병관] [오후 4:26] 어, 왜? 좀 늦었나?
긍정 -> [김경식] [오후 4:27] 아냐, 아냐. 너 나름 열심히 하네. 우리 중에 너만 그런 거 같아서 ㅋㅋ
부정 -> [민병관] [오후 4:28] 뭐야, 갑자기 왜 그런 말을 해?
부정 -> [김경식] [오후 4:29] 아니, 그냥. 다들 벌써 끝났는데 너 혼자 아직도 하고 있어서 말이지. 대단하다, 진짜.
부정 -> [민병관] [오후 4:30] 아, 다들 벌써 끝냈어? 나만 몰랐네…
부정 -> [김경식] [오후 4:31] 어, 그러게. 근데 뭐, 늦어도 괜찮아. 너는 원래 그런 스타일이니까.
부정 -> [민병관] [오후 4:32] 그래도 좀 아쉽네. 같이 끝내는 줄 알았는데.
긍정 -> [김경식] [오후 4:33] 아, 뭐 어쩔 수 없지. 다음에 우리랑 같은 속도로 할 수 있겠지? ㅋㅋ
부정 -> [민병관] [오후 4:34] …응, 다음엔 좀 더 빨리 해볼게.
부정 -> [김경식] [오후 4:35] 그래, 그래. 근데 너무 스트레스 받지 마라. 우리끼리 알아서 잘 맞춰줄 테니까.
부정 -> [민병관] [오후 4:36] 응, 고맙다…
부정 -> [김경식] [오후 4:37] 아, 근데 우리 이번 주말에 뭐 하기로 했던 거 있었는데, 너도 올 거지? 아, 근데 너 일하느라 바쁠 수도 있겠다?
부정 -> [민병관] [오후 4:38] 아냐, 나도 가려고 했어. 시간 맞출 수 있어.
부정 -> [김경식] [오후 4:39] 오, 그래? 근데 만약 못 오면 뭐, 어쩔 수 없지. 우리가 알아서 재밌게 놀게.
부정 -> [민병관] [오후 4:40] 아니, 꼭 갈게. 약속했잖아.
긍정 -> [김경식] [오후 4:41] ㅋㅋ 그래, 알겠어. 그럼 기대할게, 병관아.
사이버불링


In [15]:
# 예시 문장 리스트
sentences = [
"[김경식] [오전 11:02] 야, 민병관. 너 이번에 발표 준비 어떻게 하고 있냐? 뭔가 좀 걱정돼서 말이야.",
"[민병관] [오전 11:03] 아, 나 준비하고 있지. 너희가 걱정할 필요는 없을 거 같아.",
"[문병식] [오전 11:05] 오, 그렇구나. 근데 민병관, 너 발표할 때 목소리가 좀 작아서 잘 안 들리는 거 알지? 지난번에도 다들 잘 못 들었다고 하더라.",
"[김경식] [오전 11:06] 맞아, 그리고 네가 설명할 때 좀 복잡하게 하는 경향이 있더라. 좀 더 쉽게 얘기해주면 좋을 텐데... 뭐, 이번엔 좀 나아졌겠지?",
"[민병관] [오전 11:08] 아, 그렇구나. 이번엔 좀 더 신경 쓸게.",
"[문병식] [오전 11:10] ㅋㅋ 그래, 네가 나름 노력하는 건 알겠는데, 우리 팀 성적에 영향 줄까 봐 살짝 걱정은 된다. 다들 기대하는 수준이 있잖아.",
"[김경식] [오전 11:12] 응, 우리도 네가 잘하길 바라긴 해. 근데 너도 알잖아, 우리가 좀 중요하게 생각하는 거니까. 괜히 우리가 실망하는 일 없었으면 좋겠네.",
"[민병관] [오전 11:14] 알겠어. 최선을 다할게.",
"[문병식] [오전 11:16] 근데 민병관, 너 맨날 최선을 다한다고 하는데, 결과가 좀 아쉬운 게 많았잖아. 이번엔 진짜 잘해봐라. 우리도 네가 잘하길 진심으로 바라고 있으니까.",
"[김경식] [오전 11:18] 맞아, 네가 노력한 건 알겠어. 근데 노력만으로는 안 될 때도 있잖아. 뭐, 이번엔 잘하겠지?",
"[민병관] [오전 11:20] 아... 그래, 알겠어. 이번엔 진짜 신경 써볼게.",
"[문병식] [오전 11:22] 그래, 기대할게. 우리도 너만 믿고 있는 건 아니니까, 너도 알아서 잘하겠지."
]

# 예시 문장 처리
predict_with_cyberbullying_check(sentences, tokenizer, model, device)

부정 -> [김경식] [오전 11:02] 야, 민병관. 너 이번에 발표 준비 어떻게 하고 있냐? 뭔가 좀 걱정돼서 말이야.
부정 -> [민병관] [오전 11:03] 아, 나 준비하고 있지. 너희가 걱정할 필요는 없을 거 같아.
부정 -> [문병식] [오전 11:05] 오, 그렇구나. 근데 민병관, 너 발표할 때 목소리가 좀 작아서 잘 안 들리는 거 알지? 지난번에도 다들 잘 못 들었다고 하더라.
부정 -> [김경식] [오전 11:06] 맞아, 그리고 네가 설명할 때 좀 복잡하게 하는 경향이 있더라. 좀 더 쉽게 얘기해주면 좋을 텐데... 뭐, 이번엔 좀 나아졌겠지?
부정 -> [민병관] [오전 11:08] 아, 그렇구나. 이번엔 좀 더 신경 쓸게.
긍정 -> [문병식] [오전 11:10] ㅋㅋ 그래, 네가 나름 노력하는 건 알겠는데, 우리 팀 성적에 영향 줄까 봐 살짝 걱정은 된다. 다들 기대하는 수준이 있잖아.
부정 -> [김경식] [오전 11:12] 응, 우리도 네가 잘하길 바라긴 해. 근데 너도 알잖아, 우리가 좀 중요하게 생각하는 거니까. 괜히 우리가 실망하는 일 없었으면 좋겠네.
부정 -> [민병관] [오전 11:14] 알겠어. 최선을 다할게.
부정 -> [문병식] [오전 11:16] 근데 민병관, 너 맨날 최선을 다한다고 하는데, 결과가 좀 아쉬운 게 많았잖아. 이번엔 진짜 잘해봐라. 우리도 네가 잘하길 진심으로 바라고 있으니까.
부정 -> [김경식] [오전 11:18] 맞아, 네가 노력한 건 알겠어. 근데 노력만으로는 안 될 때도 있잖아. 뭐, 이번엔 잘하겠지?
부정 -> [민병관] [오전 11:20] 아... 그래, 알겠어. 이번엔 진짜 신경 써볼게.
부정 -> [문병식] [오전 11:22] 그래, 기대할게. 우리도 너만 믿고 있는 건 아니니까, 너도 알아서 잘하겠지.
사이버불링


In [16]:
# 예시 문장 리스트
sentences = [
"[김수혁] [오후 5:10] 야, 민병관. 너 요즘 운동 좀 하냐? 지난번에 체육 시간에 완전 힘들어하던데.",
"[민병관] [오후 5:11] 아, 요즘 좀 바빠서 못 했어. 다시 시작하려고 생각 중이야.",
"[박진우] [오후 5:13] ㅋㅋㅋ 시작만 생각하는 거지? 뭐, 너 체력으로 운동하면 곧 지치겠지.",
"[민병관] [오후 5:14] 그래도 조금씩이라도 하려고.",
"[김수혁] [오후 5:16] 조금씩? 그 정도로는 변화가 있겠냐? 우리랑 같이 뛰려면 한참 멀었겠다.",
"[민병관] [오후 5:17] 계속 하면 나아지지 않을까?",
"[박진우] [오후 5:19] 뭐, 기대는 안 하지만 너 혼자라도 해봐라. 우리가 너랑 맞춰줄 순 없으니까."
]

# 예시 문장 처리
predict_with_cyberbullying_check(sentences, tokenizer, model, device)

부정 -> [김수혁] [오후 5:10] 야, 민병관. 너 요즘 운동 좀 하냐? 지난번에 체육 시간에 완전 힘들어하던데.
부정 -> [민병관] [오후 5:11] 아, 요즘 좀 바빠서 못 했어. 다시 시작하려고 생각 중이야.
긍정 -> [박진우] [오후 5:13] ㅋㅋㅋ 시작만 생각하는 거지? 뭐, 너 체력으로 운동하면 곧 지치겠지.
부정 -> [민병관] [오후 5:14] 그래도 조금씩이라도 하려고.
부정 -> [김수혁] [오후 5:16] 조금씩? 그 정도로는 변화가 있겠냐? 우리랑 같이 뛰려면 한참 멀었겠다.
부정 -> [민병관] [오후 5:17] 계속 하면 나아지지 않을까?
부정 -> [박진우] [오후 5:19] 뭐, 기대는 안 하지만 너 혼자라도 해봐라. 우리가 너랑 맞춰줄 순 없으니까.
사이버불링


In [17]:
# 예시 문장 리스트
sentences = [
"[정유진] [오후 2:30] 민지야, 요즘 다이어트 한다고 들었는데, 힘들지 않아?",
"[신예슬] [오후 2:31] 그러게, 내가 다이어트 해봤는데 생각보다 쉽지 않더라. 민지 너는 잘 할 수 있을까?",
"[민지] [오후 2:32] 응, 좀 힘들긴 한데 열심히 해보려고.",
"[정유진] [오후 2:33] 응원할게. 근데 민지니까 하루 이틀 하다가 포기할 것 같은데?",
"[신예슬] [오후 2:34] 맞아, 너 먹는 거 좋아하잖아. 다이어트할 때 먹고 싶은 거 참기 힘들지 않아?",
"[민지] [오후 2:35] 괜찮아, 잘 참을 수 있을 거야.",
"[정유진] [오후 2:36] 그래도 우리끼리 맛있는 거 먹을 때 민지 생각날 것 같아. 참는 모습 너무 귀엽겠다."
]

# 예시 문장 처리
predict_with_cyberbullying_check(sentences, tokenizer, model, device)

부정 -> [정유진] [오후 2:30] 민지야, 요즘 다이어트 한다고 들었는데, 힘들지 않아?
부정 -> [신예슬] [오후 2:31] 그러게, 내가 다이어트 해봤는데 생각보다 쉽지 않더라. 민지 너는 잘 할 수 있을까?
부정 -> [민지] [오후 2:32] 응, 좀 힘들긴 한데 열심히 해보려고.
부정 -> [정유진] [오후 2:33] 응원할게. 근데 민지니까 하루 이틀 하다가 포기할 것 같은데?
부정 -> [신예슬] [오후 2:34] 맞아, 너 먹는 거 좋아하잖아. 다이어트할 때 먹고 싶은 거 참기 힘들지 않아?
부정 -> [민지] [오후 2:35] 괜찮아, 잘 참을 수 있을 거야.
부정 -> [정유진] [오후 2:36] 그래도 우리끼리 맛있는 거 먹을 때 민지 생각날 것 같아. 참는 모습 너무 귀엽겠다.
사이버불링


In [18]:
# 예시 문장 리스트
sentences = [
"[최민준] [오후 4:00] 와, 수현아 오늘도 혼자서 밥 먹더라? 진짜 멋있어, 요즘 트렌드를 선도하는 거지? ㅋㅋ",
"[박지훈] [오후 4:01] 맞아, 요즘 혼밥이 대세잖아. 근데 매일 혼자 먹는 건 좀 외롭지 않아?",
"[김하윤] [오후 4:02] 나도 수현이처럼 혼자 밥 먹고 싶다. 근데 아무도 같이 먹자는 사람 없으면 좀 슬플 것 같아 ㅋㅋ",
"[최민준] [오후 4:03] 아, 수현이는 그런 거 신경 안 쓸걸? 워낙 독립적인 사람이잖아. 자기만의 시간이 소중한 거겠지",
"[박지훈] [오후 4:04] 그치 그치. 수현이처럼 강한 사람은 혼자 있는 게 편할 거야. 우리가 괜히 같이 먹자고 하면 불편해질 수도 있어 ㅋㅋ",
"[김하윤] [오후 4:05] 맞아, 수현이는 자기만의 세계가 확고한 것 같아. 근데 솔직히 말해봐, 좀 외롭진 않니?",
"[최민준] [오후 4:06] 나 같으면 솔직히 친구들한테 외면당한 것 같아서 못 할 것 같은데... 수현이는 진짜 대단해",
"[박지훈] [오후 4:07] 응, 매일 혼자서도 잘 다니는 거 보면 정말 강한 멘탈이야. 우리가 배워야 할 점이 많아 ㅋㅋ",
"[김하윤] [오후 4:08] 그리고 오늘 체육 시간에 수현이가 혼자 스트레칭하는 거 봤는데, 정말 집중 잘하더라. 다른 애들하고는 아예 안 어울리고 자기만의 운동 루틴이 있는 건가? ㅋㅋ",
"[최민준] [오후 4:09] 와, 진짜 프로페셔널하다. 근데 혹시 그냥 같이 할 친구가 없어서 그런 건 아니지? ㅋㅋㅋ",
"[박지훈] [오후 4:10] 설마~ 수현이 정도면 친구들이 먼저 같이 하자고 할 텐데, 아마 자기만의 시간이 중요한 거겠지",
"[김하윤] [오후 4:11] 그렇겠지? 근데 오늘 우리 다 같이 체육 끝나고 얘기하는데, 수현이는 그냥 조용히 혼자 갔더라? 우리랑 이야기하는 게 싫은 걸까?",
"[최민준] [오후 4:12] 아, 수현이는 그런 소소한 대화엔 관심 없을걸? 너무 높은 수준이라 우리랑 안 맞을 수도 있어 ㅋㅋ",
"[박지훈] [오후 4:13] ㅋㅋㅋㅋ 그럴지도 몰라. 우리끼리 떠드는 거보다 혼자 있는 게 더 좋을 수도 있지",
"[김하윤] [오후 4:14] 맞아, 수현이처럼 혼자 있는 거 잘하는 사람 보기 드문데. 근데 가끔은 좀 외로울 수도 있지 않을까?",
"[최민준] [오후 4:15] 외로움? 수현이는 그런 거 모르겠지. 워낙 멘탈이 강해서 그런 거 신경도 안 쓸 거야",
"[박지훈] [오후 4:16] 수현아, 진짜 대단하다. 우리라면 상상도 못할 일인데 말이야. 혼자서도 잘 지내는 거 보면 진짜 존경스러워 ㅋㅋ",
"[김하윤] [오후 4:17] 그러게, 수현이는 진짜 특별해. 우리처럼 평범한 애들은 따라할 수 없는 거지",
"[최민준] [오후 4:18] ㅋㅋㅋ 맞아, 수현이처럼 혼자서도 잘 지내는 사람은 진짜 보기 드물어. 우리도 좀 배워야겠어",
"[박지훈] [오후 4:19] 그러니까. 근데 가끔은 친구랑도 어울리는 것도 좋을 텐데... 수현이는 그냥 그게 싫은 건가?",
"[김하윤] [오후 4:20] 뭐, 그럴 수도 있지. 어쨌든 수현아, 우리랑 같이 어울리는 것도 좋으니까, 언제든지 말해줘 ㅋㅋ",
"[최민준] [오후 4:21] 맞아, 우리끼리도 좋지만, 가끔은 같이 하는 것도 나쁘지 않을 거야. 언제든지 불러줘 ㅋㅋ"
]

# 예시 문장 처리
predict_with_cyberbullying_check(sentences, tokenizer, model, device)

긍정 -> [최민준] [오후 4:00] 와, 수현아 오늘도 혼자서 밥 먹더라? 진짜 멋있어, 요즘 트렌드를 선도하는 거지? ㅋㅋ
부정 -> [박지훈] [오후 4:01] 맞아, 요즘 혼밥이 대세잖아. 근데 매일 혼자 먹는 건 좀 외롭지 않아?
긍정 -> [김하윤] [오후 4:02] 나도 수현이처럼 혼자 밥 먹고 싶다. 근데 아무도 같이 먹자는 사람 없으면 좀 슬플 것 같아 ㅋㅋ
부정 -> [최민준] [오후 4:03] 아, 수현이는 그런 거 신경 안 쓸걸? 워낙 독립적인 사람이잖아. 자기만의 시간이 소중한 거겠지
긍정 -> [박지훈] [오후 4:04] 그치 그치. 수현이처럼 강한 사람은 혼자 있는 게 편할 거야. 우리가 괜히 같이 먹자고 하면 불편해질 수도 있어 ㅋㅋ
부정 -> [김하윤] [오후 4:05] 맞아, 수현이는 자기만의 세계가 확고한 것 같아. 근데 솔직히 말해봐, 좀 외롭진 않니?
부정 -> [최민준] [오후 4:06] 나 같으면 솔직히 친구들한테 외면당한 것 같아서 못 할 것 같은데... 수현이는 진짜 대단해
긍정 -> [박지훈] [오후 4:07] 응, 매일 혼자서도 잘 다니는 거 보면 정말 강한 멘탈이야. 우리가 배워야 할 점이 많아 ㅋㅋ
긍정 -> [김하윤] [오후 4:08] 그리고 오늘 체육 시간에 수현이가 혼자 스트레칭하는 거 봤는데, 정말 집중 잘하더라. 다른 애들하고는 아예 안 어울리고 자기만의 운동 루틴이 있는 건가? ㅋㅋ
긍정 -> [최민준] [오후 4:09] 와, 진짜 프로페셔널하다. 근데 혹시 그냥 같이 할 친구가 없어서 그런 건 아니지? ㅋㅋㅋ
부정 -> [박지훈] [오후 4:10] 설마~ 수현이 정도면 친구들이 먼저 같이 하자고 할 텐데, 아마 자기만의 시간이 중요한 거겠지
부정 -> [김하윤] [오후 4:11] 그렇겠지? 근데 오늘 우리 다 같이 체육 끝나고 얘기하는데, 수현이는 그냥 조용히 혼자 갔더라? 우리랑 이야기하는 게 싫은 걸까?
긍정 -> [최민준] [오후 4:12] 아, 수현이는 그런 소소한 대화엔 관심 없을걸? 

In [ ]:
# 일상대화

In [ ]:
#친구랑 욕설 (사이버불링 x)

In [19]:
# 예시 문장 리스트
sentences = [
"[김경식] [오후 4:25] 야, 이 개새끼야. 어디 갔다 왔냐?",
"[민병관] [오후 4:26] 어이, 미친놈아. 그냥 밖에 바람 좀 쐬고 왔다. 왜?",
"[김경식] [오후 4:27] 아, 씨… 너가 없으니까 심심해서 그렇지. 왜 그렇게 잠수타고 다녀?",
"[민병관] [오후 4:28] 에이, 잠수는 무슨. 그냥 전화하면 바로 받지, 새끼야.",
"[김경식] [오후 4:29] 아, 진짜 한 번만 더 그러면 내가 찾아간다. ㅋㅋ",
"[민병관] [오후 4:30] 어이, 그럼 내가 도망갈 줄 아냐? 와서 내가 요리하는 거나 구경해라.",
"[김경식] [오후 4:31] 너 요리도 못 하면서 잘난 척은 ㅋㅋ 뭐 만들 건데?",
"[민병관] [오후 4:32] 아, 이 새끼. 너 놀리려고 한 건데, 갑자기 왜 진지해져?",
"[김경식] [오후 4:33] ㅋㅋ 그러니까 말이야. 그냥 너한테 얻어먹으려던 건데.",
"[민병관] [오후 4:34] 야, 그러면 미리 말을 하던가. 내가 준비 다 해놓지. 너는 그냥 빈손으로 오면 되잖아.",
"[김경식] [오후 4:35] 와, 감동이다. 이 새끼가 그래도 친구는 친구다. 그럼 뭐 필요한 거 있으면 말해라.",
"[민병관] [오후 4:36] 응, 필요하면 말하지 뭐. 너만 있으면 됐다, 이 자식아.",
"[김경식] [오후 4:37] 아, 씨… 감동 먹을 뻔했네. ㅋㅋ 그래, 곧 갈 테니까 준비해라.",
"[민병관] [오후 4:38] 오케이, 새끼야. 오기만 해라. 같이 재밌게 놀자."

]

# 예시 문장 처리
predict_with_cyberbullying_check(sentences, tokenizer, model, device)

긍정 -> [김경식] [오후 4:25] 야, 이 개새끼야. 어디 갔다 왔냐?
긍정 -> [민병관] [오후 4:26] 어이, 미친놈아. 그냥 밖에 바람 좀 쐬고 왔다. 왜?
긍정 -> [김경식] [오후 4:27] 아, 씨… 너가 없으니까 심심해서 그렇지. 왜 그렇게 잠수타고 다녀?
긍정 -> [민병관] [오후 4:28] 에이, 잠수는 무슨. 그냥 전화하면 바로 받지, 새끼야.
긍정 -> [김경식] [오후 4:29] 아, 진짜 한 번만 더 그러면 내가 찾아간다. ㅋㅋ
부정 -> [민병관] [오후 4:30] 어이, 그럼 내가 도망갈 줄 아냐? 와서 내가 요리하는 거나 구경해라.
긍정 -> [김경식] [오후 4:31] 너 요리도 못 하면서 잘난 척은 ㅋㅋ 뭐 만들 건데?
긍정 -> [민병관] [오후 4:32] 아, 이 새끼. 너 놀리려고 한 건데, 갑자기 왜 진지해져?
긍정 -> [김경식] [오후 4:33] ㅋㅋ 그러니까 말이야. 그냥 너한테 얻어먹으려던 건데.
부정 -> [민병관] [오후 4:34] 야, 그러면 미리 말을 하던가. 내가 준비 다 해놓지. 너는 그냥 빈손으로 오면 되잖아.
긍정 -> [김경식] [오후 4:35] 와, 감동이다. 이 새끼가 그래도 친구는 친구다. 그럼 뭐 필요한 거 있으면 말해라.
긍정 -> [민병관] [오후 4:36] 응, 필요하면 말하지 뭐. 너만 있으면 됐다, 이 자식아.
긍정 -> [김경식] [오후 4:37] 아, 씨… 감동 먹을 뻔했네. ㅋㅋ 그래, 곧 갈 테니까 준비해라.
긍정 -> [민병관] [오후 4:38] 오케이, 새끼야. 오기만 해라. 같이 재밌게 놀자.
사이버불링이 아님


In [20]:
# 예시 문장 리스트
sentences = [
"[최현준] [오후 5:30] 야, 민수야. 니가 그렇게 뛰니까 ㅈㄴ 느리잖아. 개달팽이냐?",
"[김민수] [오후 5:31] ㅅㅂ 너나 잘해. 너도 숨차서 뒤쳐지던데ㅋㅋ",
"[박상혁] [오후 5:33] 둘 다 입만 살았네. 니들 이래서 체력 키우긴 글렀다.",
"[최현준] [오후 5:34] 상혁아, 니가 제일 못 뛰던데? 입 좀 닥치고 뛰어라ㅋㅋ",
"[김민수] [오후 5:35] 그러게 말이다. 니가 우리한테 큰소리 칠 입장이냐? 이 ㅅㄲ야",
"[박상혁] [오후 5:36] 그래, 다음엔 니들 다 따돌릴 거다. 이 ㅂㅅ들아, 각오해라!",
"[최현준] [오후 5:37] 웃기고 있네. 니가 그동안 계속 우리한테 뒤쳐졌잖아ㅋㅋ",
"[김민수] [오후 5:38] 그래놓고 개뿔. 다음에 또 너 제일 먼저 지쳐서 나가떨어질 거면서ㅋㅋ",
"[박상혁] [오후 5:39] 됐고, 내일 체력 테스트나 하자. 누가 진짜 ㅂㅅ인지 가리자.",
"[최현준] [오후 5:40] 그래, 내일 한번 해보자. 누가 지치나 보자고, 이 ㅅㄲ들아!"

]

# 예시 문장 처리
predict_with_cyberbullying_check(sentences, tokenizer, model, device)

긍정 -> [최현준] [오후 5:30] 야, 민수야. 니가 그렇게 뛰니까 ㅈㄴ 느리잖아. 개달팽이냐?
긍정 -> [김민수] [오후 5:31] ㅅㅂ 너나 잘해. 너도 숨차서 뒤쳐지던데ㅋㅋ
부정 -> [박상혁] [오후 5:33] 둘 다 입만 살았네. 니들 이래서 체력 키우긴 글렀다.
긍정 -> [최현준] [오후 5:34] 상혁아, 니가 제일 못 뛰던데? 입 좀 닥치고 뛰어라ㅋㅋ
긍정 -> [김민수] [오후 5:35] 그러게 말이다. 니가 우리한테 큰소리 칠 입장이냐? 이 ㅅㄲ야
긍정 -> [박상혁] [오후 5:36] 그래, 다음엔 니들 다 따돌릴 거다. 이 ㅂㅅ들아, 각오해라!
긍정 -> [최현준] [오후 5:37] 웃기고 있네. 니가 그동안 계속 우리한테 뒤쳐졌잖아ㅋㅋ
긍정 -> [김민수] [오후 5:38] 그래놓고 개뿔. 다음에 또 너 제일 먼저 지쳐서 나가떨어질 거면서ㅋㅋ
긍정 -> [박상혁] [오후 5:39] 됐고, 내일 체력 테스트나 하자. 누가 진짜 ㅂㅅ인지 가리자.
긍정 -> [최현준] [오후 5:40] 그래, 내일 한번 해보자. 누가 지치나 보자고, 이 ㅅㄲ들아!
사이버불링이 아님


In [21]:
# 예시 문장 리스트
sentences = [
"[정수현] [오후 9:00] 야, 이거 문제 ㅈㄴ 어렵네. 도대체 무슨 말인지 모르겠다.",
"[박진수] [오후 9:01] 니가 공부를 ㅂㅅ같이 하니까 그렇지. 개념 좀 봐라.",
"[이동민] [오후 9:03] ㅅㅂ 나도 모르겠는데. 만든 사람 머리에 돌 들어있나 봐ㅋㅋ",
"[정수현] [오후 9:04] 야, 돌이라니. 진수 니 머리가 문제지. 내 문제 아니거든?",
"[박진수] [오후 9:05] 개소리 말고 집중이나 해라. 또 점수 꼴찌하겠다 이놈들아ㅋㅋ",
"[이동민] [오후 9:06] 야, 너나 신경 써라. 너도 맨날 공부할 때 삽질하잖아.",
"[정수현] [오후 9:07] ㅋㅋㅋ 너네 둘 다 공부 ㅂㅅ이라서 그런 거 아님? 나도 이해 안 간다.",
"[박진수] [오후 9:08] 어이없네. 내가 이 중에 제일 잘하는 거 모르냐? 너희랑 다르다고.",
"[이동민] [오후 9:09] ㅋㅋ 그러니까 큰소리 치지 말고 푼 거 보여줘봐라. 아니면 우리 셋 다 포기하든가.",
"[정수현] [오후 9:10] 좋다, 우리 그냥 다 포기하고 피시방이나 갈래? 이 ㅅㄲ들아, 공부 안 맞는 거 인정해라."

]

# 예시 문장 처리
predict_with_cyberbullying_check(sentences, tokenizer, model, device)

긍정 -> [정수현] [오후 9:00] 야, 이거 문제 ㅈㄴ 어렵네. 도대체 무슨 말인지 모르겠다.
긍정 -> [박진수] [오후 9:01] 니가 공부를 ㅂㅅ같이 하니까 그렇지. 개념 좀 봐라.
긍정 -> [이동민] [오후 9:03] ㅅㅂ 나도 모르겠는데. 만든 사람 머리에 돌 들어있나 봐ㅋㅋ
부정 -> [정수현] [오후 9:04] 야, 돌이라니. 진수 니 머리가 문제지. 내 문제 아니거든?
긍정 -> [박진수] [오후 9:05] 개소리 말고 집중이나 해라. 또 점수 꼴찌하겠다 이놈들아ㅋㅋ
부정 -> [이동민] [오후 9:06] 야, 너나 신경 써라. 너도 맨날 공부할 때 삽질하잖아.
긍정 -> [정수현] [오후 9:07] ㅋㅋㅋ 너네 둘 다 공부 ㅂㅅ이라서 그런 거 아님? 나도 이해 안 간다.
부정 -> [박진수] [오후 9:08] 어이없네. 내가 이 중에 제일 잘하는 거 모르냐? 너희랑 다르다고.
긍정 -> [이동민] [오후 9:09] ㅋㅋ 그러니까 큰소리 치지 말고 푼 거 보여줘봐라. 아니면 우리 셋 다 포기하든가.
긍정 -> [정수현] [오후 9:10] 좋다, 우리 그냥 다 포기하고 피시방이나 갈래? 이 ㅅㄲ들아, 공부 안 맞는 거 인정해라.
사이버불링이 아님


In [22]:
# 예시 문장 리스트
sentences = [
"[김경식] [오전 12:10] 야, ㅅㅂ 오늘 영어쌤 또 지랄하더라. 숙제 안 했다고 나만 또 찝어냈어.",
"[민병관] [오전 12:12] ㅋㅋㅋ 네가 또 까먹고 안 했냐? 맨날 그러니까 쌤이 너만 기억하잖아ㅋㅋㅋㅄ. 야, 근데 나도 그 숙제 ㅈㄴ 헷갈렸음.",
"[문병식] [오전 12:14] ㄹㅇ, 그거 존나 어렵더라. 나도 겨우 했어. 근데 김경식, 너 진짜 대단하다. 맨날 혼나면서도 정신 못 차리냐? ㅋㅋ",
"[김경식] [오전 12:16] ㅋㅋㅋ 야, 나도 할 말 많거든. 근데 왜 맨날 나만 걸리냐고. 진짜 재수 없게ㅅㅂ.",
"[민병관] [오전 12:18] ㄹㅇ 개재수 없는 날인 듯. 나도 오늘 체육시간에 개같이 뛰었더니 다리 아파 죽겠네.",
"[문병식] [오전 12:20] ㅋㅋㅋ 다들 고생하네. 야, 근데 체육시간에 김경식은 또 뒤쳐졌지? 맨날 꼴찌더라. 너새끼는 체력 좀 길러라.",
"[김경식] [오전 12:22] ㅅㅂ 나도 알거든. 근데 체육은 그냥 적당히 하는 거지 뭐 그렇게 열심히 하냐? 힘들어 죽겠는데.",
"[민병관] [오전 12:24] 그래도 좀 열심히 해라, 병신아. 맨날 너만 느려 터져서 우리까지 지적 받잖아. 쌤한테 까이기 싫으면.",
"[문병식] [오전 12:26] ㅋㅋㅋㅈㄹ 민병관 말이 맞음. 야, 다음번엔 좀 빨리 뛰어라. 아니면 우리 전부 걸려서 같이 혼날 듯.",
"[김경식] [오전 12:28] ㅇㅋ, 알겠음. 다음엔 좀 더 빨리 뛰어볼게. 근데 오늘 끝나고 뭐하냐? PC방 ㄱ?",
"[민병관] [오전 12:30] 좋지, ㅅㅂ 나 오늘 스트레스 풀어야겠음. 같이 롤 한 판 돌리자.",
"[문병식] [오전 12:32] ㅇㅋ, 그러면 학교 끝나고 바로 가자. 간만에 고기나 쳐먹고 드가자."
]

# 예시 문장 처리
predict_with_cyberbullying_check(sentences, tokenizer, model, device)

긍정 -> [김경식] [오전 12:10] 야, ㅅㅂ 오늘 영어쌤 또 지랄하더라. 숙제 안 했다고 나만 또 찝어냈어.
긍정 -> [민병관] [오전 12:12] ㅋㅋㅋ 네가 또 까먹고 안 했냐? 맨날 그러니까 쌤이 너만 기억하잖아ㅋㅋㅋㅄ. 야, 근데 나도 그 숙제 ㅈㄴ 헷갈렸음.
긍정 -> [문병식] [오전 12:14] ㄹㅇ, 그거 존나 어렵더라. 나도 겨우 했어. 근데 김경식, 너 진짜 대단하다. 맨날 혼나면서도 정신 못 차리냐? ㅋㅋ
긍정 -> [김경식] [오전 12:16] ㅋㅋㅋ 야, 나도 할 말 많거든. 근데 왜 맨날 나만 걸리냐고. 진짜 재수 없게ㅅㅂ.
긍정 -> [민병관] [오전 12:18] ㄹㅇ 개재수 없는 날인 듯. 나도 오늘 체육시간에 개같이 뛰었더니 다리 아파 죽겠네.
긍정 -> [문병식] [오전 12:20] ㅋㅋㅋ 다들 고생하네. 야, 근데 체육시간에 김경식은 또 뒤쳐졌지? 맨날 꼴찌더라. 너새끼는 체력 좀 길러라.
긍정 -> [김경식] [오전 12:22] ㅅㅂ 나도 알거든. 근데 체육은 그냥 적당히 하는 거지 뭐 그렇게 열심히 하냐? 힘들어 죽겠는데.
긍정 -> [민병관] [오전 12:24] 그래도 좀 열심히 해라, 병신아. 맨날 너만 느려 터져서 우리까지 지적 받잖아. 쌤한테 까이기 싫으면.
긍정 -> [문병식] [오전 12:26] ㅋㅋㅋㅈㄹ 민병관 말이 맞음. 야, 다음번엔 좀 빨리 뛰어라. 아니면 우리 전부 걸려서 같이 혼날 듯.
긍정 -> [김경식] [오전 12:28] ㅇㅋ, 알겠음. 다음엔 좀 더 빨리 뛰어볼게. 근데 오늘 끝나고 뭐하냐? PC방 ㄱ?
긍정 -> [민병관] [오전 12:30] 좋지, ㅅㅂ 나 오늘 스트레스 풀어야겠음. 같이 롤 한 판 돌리자.
긍정 -> [문병식] [오전 12:32] ㅇㅋ, 그러면 학교 끝나고 바로 가자. 간만에 고기나 쳐먹고 드가자.
사이버불링이 아님
